# NEX-GDDP-CMIP6 Download + Multivariate Bias Correction & Downscaling
## Using HYRAS as reference (1951–1980) · Variables: `pr`, `tas`, `tasmax`, `tasmin`, `rsds`, `hurs`

**Workflow overview:**
1. Download NEX-GDDP-CMIP6 historical & future data via `climdata`
2. Load HYRAS reference observations
3. Align grids (regrid NEX-GDDP → HYRAS grid)
4. Slice to reference period 1951–1980
5. Train multivariate bias correction (MBCn) using `xclim.sdba`
6. Apply correction to full simulation period
7. Post-process & validate outputs
8. Visualise results

## 1 · Import Required Libraries

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import iris

# climdata for data download
from climdata import ClimData

# climdata BCSD pipeline (wraps ISIMIP3BASD)
from climdata.sdba.bcsd import (
    BiasCorrection,
    StatisticalDownscaling,
    regrid_to_coarse,
)

print("✓ All imports successful")

import dask
import dask.array as da

# ── Dask global config ────────────────────────────────────────────────────────
dask.config.set({
    "scheduler": "synchronous",   # safe for netcdf4 + fork()
    "array.chunk-size": "256MiB",
})

print("✓ Dask config: scheduler=synchronous (thread-safe for netcdf4 + multiprocessing)")


✓ All imports successful
✓ Dask config: scheduler=synchronous (thread-safe for netcdf4 + multiprocessing)


## 2 · Configuration & Parameters

In [2]:
# ── Variables ────────────────────────────────────────────────────────────────
VARIABLES = ["pr", "hurs"]#, "tas", "tasmax", "tasmin", "rsds", "hurs"]

# ── Periods ──────────────────────────────────────────────────────────────────
REF_START   = "1951-01-01"   # HYRAS reference period start
REF_END     = "1980-12-31"   # HYRAS reference period end

HIST_START  = "1951-01-01"   # NEX-GDDP historical download start
HIST_END    = "2014-12-31"   # NEX-GDDP historical download end

FUT_START   = "2015-01-01"   # NEX-GDDP future download start
FUT_END     = "2100-12-31"   # NEX-GDDP future download end

# ── NEX-GDDP model & scenario ─────────────────────────────────────────────────
SOURCE_ID    = "GFDL-ESM4"   # CMIP6 model name
MEMBER_ID    = "r1i1p1f1"
SCENARIO     = "ssp370"          # future scenario

# ── Region (Germany / Central Europe) ─────────────────────────────────────────
REGION = "germany"

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = "/data01/FDS/muduchuru/Atmos/"
OUTPUT_DIR = os.path.join(DATA_DIR, "NEXGDDP_HYRAS_ISIMIP3BASD")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"  Variables  : {VARIABLES}")
print(f"  Ref period : {REF_START} → {REF_END}")
print(f"  Model      : {SOURCE_ID} / {MEMBER_ID}")
print(f"  Scenario   : {SCENARIO}")
print(f"  Output dir : {OUTPUT_DIR}")

Configuration:
  Variables  : ['pr', 'hurs']
  Ref period : 1951-01-01 → 1980-12-31
  Model      : GFDL-ESM4 / r1i1p1f1
  Scenario   : ssp370
  Output dir : /data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_ISIMIP3BASD


## 3 · Download NEX-GDDP-CMIP6 Data

Download both **historical** (1951–2014) and **future** scenario data via `climdata`'s `ClimData` extractor.  
NEX-GDDP provides statistically downscaled CMIP6 projections at **0.25°** (~25 km) globally.

In [3]:

import copy
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def load(extractor, dataset, start, end, experiment=None, cache_path=None, force=False):
    if cache_path and os.path.exists(cache_path) and not force:
        print(f"  📂 Loading from cache: {cache_path}")
        return xr.open_dataset(cache_path)

    # work on a private copy so concurrent threads don't share mutable cfg
    ext = copy.deepcopy(extractor)
    ext.cfg.dataset = dataset
    ext.cfg.time_range.start_date = start
    ext.cfg.time_range.end_date = end
    if dataset == "hyras":
        ext.cfg.data_dir = ext.cfg.data_dir + 'DWD'
    if experiment is not None:
        ext.cfg.experiment_id = experiment
    ds = ext.extract()

    if cache_path:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        ds.to_netcdf(cache_path)
        print(f"  💾 Saved to cache: {cache_path}")

    return ds

FORCE_RELOAD = False   # set True to re-download and overwrite cached files

print("Loading all 3 datasets in parallel ...")
t0 = time.time()

overrides = [
    f"region={REGION}",
    f"variables={VARIABLES}",
    f"data_dir={DATA_DIR}",
    f"source_id={SOURCE_ID}",
]
extractor = ClimData(overrides=overrides)

_cache = os.path.join(OUTPUT_DIR, "cache")
tasks = {
    "sim_hist": dict(extractor=extractor, dataset='nexgddp', start=HIST_START, end=HIST_END,
                     experiment='historical',
                    #  cache_path=os.path.join(_cache, f"sim_hist_{SOURCE_ID}_{HIST_START[:4]}-{HIST_END[:4]}.nc"),
                     force=FORCE_RELOAD),
    "sim_fut":  dict(extractor=extractor, dataset='nexgddp', start=FUT_START, end=FUT_END,
                     experiment=SCENARIO,
                    #  cache_path=os.path.join(_cache, f"sim_fut_{SOURCE_ID}_{SCENARIO}_{FUT_START[:4]}-{FUT_END[:4]}.nc"),
                     force=FORCE_RELOAD),
    "obs_raw":  dict(extractor=extractor, dataset='hyras', start=REF_START, end=REF_END,
                    #  cache_path=os.path.join(_cache, f"obs_raw_HYRAS_{REF_START[:4]}-{REF_END[:4]}.nc"),
                     force=FORCE_RELOAD),
}

results = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(load, **kwargs): name for name, kwargs in tasks.items()}
    for fut in as_completed(futures):
        name = futures[fut]
        try:
            results[name] = fut.result()
        except Exception as exc:
            print(f"  ⚠️  {name} failed: {exc}")
            raise

sim_hist = results["sim_hist"]
sim_fut  = results["sim_fut"]
obs_raw  = results["obs_raw"]

print(f"\n✅ All datasets ready in {time.time() - t0:.1f}s")
print(sim_hist)
print(sim_fut)
print(obs_raw)


Loading all 3 datasets in parallel ...
🔍 Auto-discovering metadata for GFDL-ESM4/historical...
   Checking available realizations...
🔍 Auto-discovering metadata for GFDL-ESM4/ssp370...
   Checking available realizations...
⬇️  Checking: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/precipitation/pr_hyras_1_1951_v6-1_de.nc
✔️  Exists locally: /data01/FDS/muduchuru/Atmos/DWD/HYRAS/PR/pr_hyras_1_1951_v6-1_de.nc
⬇️  Checking: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/precipitation/pr_hyras_1_1952_v6-1_de.nc
✔️  Exists locally: /data01/FDS/muduchuru/Atmos/DWD/HYRAS/PR/pr_hyras_1_1952_v6-1_de.nc
⬇️  Checking: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/precipitation/pr_hyras_1_1953_v6-1_de.nc
✔️  Exists locally: /data01/FDS/muduchuru/Atmos/DWD/HYRAS/PR/pr_hyras_1_1953_v6-1_de.nc
⬇️  Checking: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/hyras_de/precipitation/pr_hyras_1_1

## 4 · Load & Preprocess HYRAS Reference Dataset

HYRAS provides gridded observations for Germany at **5 km** resolution.  
We load all six variables and standardise units to match NEX-GDDP conventions.

In [4]:

# ── Datasets are already in RAM (load() uses ThreadPoolExecutor + .load()) ───
# NEXGDDP.load() now opens each annual file in a thread, clips to region via
# _extract_preprocess, and calls ds.load() inside the thread — returning plain
# numpy arrays.  No dask arrays, no .compute(), no unify_chunks() needed.

import os, psutil

def _get_nbytes(ds):
    total = 0
    for v in ds.data_vars:
        arr = ds[v]
        total += arr.size * arr.dtype.itemsize
    return total

available_gb = psutil.virtual_memory().available / 1e9
total_bytes  = sum(_get_nbytes(ds) for ds in [sim_hist, sim_fut, obs_raw])
needed_gb    = total_bytes / 1e9

print("Datasets already in RAM (ThreadPoolExecutor load, no dask):")
for name, ds in [("sim_hist", sim_hist), ("sim_fut", sim_fut), ("obs_raw", obs_raw)]:
    nb = _get_nbytes(ds)
    print(f"  {name:<10} {nb/1e6:>6.1f} MB   dims={dict(ds.dims)}")
print(f"\n  Available RAM : {available_gb:.1f} GB  |  Used : {needed_gb:.3f} GB")

# Derive reference-period slices and coarsened obs — pure numpy ops
obs_coarse = regrid_to_coarse(obs_raw, sim_hist, method='bilinear')
sim_ref    = sim_hist.sel(time=slice(REF_START, REF_END))
obs_ref    = obs_coarse.sel(time=slice(REF_START, REF_END))

print("\n✓ Ready for bias correction")
for name, ds in [("sim_ref", sim_ref), ("obs_ref", obs_ref)]:
    print(f"  {name:<10} { {v: ds[v].shape for v in ds.data_vars} }")


Datasets already in RAM (ThreadPoolExecutor load, no dask):
  sim_hist    228.7 MB   dims={'time': 23360, 'lat': 34, 'lon': 36}
  sim_fut     307.4 MB   dims={'time': 31390, 'lat': 34, 'lon': 36}
  obs_raw    72442.5 MB   dims={'time': 10958, 'y': 890, 'x': 619}

  Available RAM : 192.2 GB  |  Used : 72.979 GB
🔄 Regridding from fine to coarse resolution using xesmf...
   Fine grid: FrozenMappingWarningOnValuesAccess({'time': 10958, 'y': 890, 'x': 619})
   Target coarse grid: FrozenMappingWarningOnValuesAccess({'time': 23360, 'lat': 34, 'lon': 36})
   Creating bilinear regridder...
   Regridding variable: pr
   Regridding variable: hurs
   ✅ Regridding complete!

✓ Ready for bias correction
  sim_ref    {'pr': (10950, 34, 36), 'hurs': (10950, 34, 36)}
  obs_ref    {'pr': (10958, 34, 36), 'hurs': (10958, 34, 36)}


In [5]:

# ── Multivariate Bias Correction (MBCn) ──────────────────────────────────────
# All variables are corrected JOINTLY in one call.
# n_iterations > 0 activates the MBCn copula rotation that preserves the
# inter-variable dependence structure (e.g. pr–hurs correlation).
#
# BiasCorrection.correct(obs_hist, sim_hist, sim_fut) → merged xr.Dataset
#   obs_hist : coarse HYRAS, reference period only (1951–1980)
#   sim_hist : NEX-GDDP historical, reference period only
#   sim_fut  : full NEX-GDDP future (2015–2100) – entire period is corrected

bc = BiasCorrection(
    variable=VARIABLES,      # pass the full variable list → multivariate MBCn
    n_iterations=20,         # MBCn rotations; 0 = univariate quantile mapping only
    n_processes=32,
    randomization_seed=42,
)

bc_result = bc.correct(
    obs_hist=obs_ref,        # coarse HYRAS, 1951–1980
    sim_hist=sim_ref,        # NEX-GDDP historical, 1951–1980
    sim_fut=sim_fut,         # NEX-GDDP future,  2015–2100
    output_path=os.path.join(OUTPUT_DIR, f"{SOURCE_ID}_{MEMBER_ID}_{SCENARIO}_BC.nc"),
)

print("\n✅ Multivariate bias correction complete")
print(bc_result)


🔧 BiasCorrection initialized for ['pr', 'hurs']
   n_iterations : 20  (MBCn multivariate)
   n_processes  : 32
   randomization_seed: 42
   [pr] dist=gamma  trend=mixed  detrend=False  adjust_p=True
   [hurs] dist=beta  trend=bounded  detrend=False  adjust_p=True

🔄 Starting bias correction for 2 variables (MBCn multivariate) ...
   Obs hist period: 1951-01-01T00:00:00.000000000 → 1980-12-31T00:00:00.000000000
   Sim hist period: 1951-01-01 12:00:00 → 1980-12-31 12:00:00
   Sim fut  period: 2015-01-01 12:00:00 → 2100-12-31 12:00:00
   Converting xarray datasets to iris cubes (in-memory)...
   [pr] cubes materialised (numpy, proleptic_gregorian) — obs_hist (10958, 34, 36), sim_hist (10950, 34, 36), sim_fut  (31390, 34, 36)
   [hurs] cubes materialised (numpy, proleptic_gregorian) — obs_hist (10958, 34, 36), sim_hist (10950, 34, 36), sim_fut  (31390, 34, 36)
   Running ISIMIP3BASD bias adjustment (n_iterations=20) ...
   (This may take a while for large datasets)
adjusting at location ..

## 6 · Statistical Downscaling (coarse 0.25° → fine 5 km HYRAS grid)

Apply **ISIMIP3BASD modified MBCn** spatial downscaling per variable.

- `obs_fine`   = raw HYRAS at 5 km (reference period 1951–1980), teaches fine-scale spatial patterns  
- `sim_coarse` = bias-corrected NEX-GDDP from Step 5 (coarse grid, full future period 2015–2100)


In [ ]:
# # ── Load bias-corrected result from disk (skip re-running BC) ─────────────────
# bc_path = os.path.join(OUTPUT_DIR, f"{SOURCE_ID}_{MEMBER_ID}_{SCENARIO}_BC.nc")
# bc_result = xr.open_dataset(bc_path)
# print(f"✓ Loaded bc_result from {bc_path}")
# print(bc_result)


: 

In [ ]:
import time
import os

# ── Load bias-corrected result from disk ──────────────────────────────────────
bc_path = os.path.join(OUTPUT_DIR, f"{SOURCE_ID}_{MEMBER_ID}_{SCENARIO}_BC.nc")
bc_result = xr.open_dataset(bc_path)
print(f"✓ Loaded bc_result from {bc_path}")
print(f"  Time range: {bc_result.time.values[0]} to {bc_result.time.values[-1]}")
print(f"  Shape: {bc_result[VARIABLES[0]].shape}")

# ── Statistical Downscaling: YEAR-BY-YEAR loop ────────────────────────────────
# The refactored StatisticalDownscaling class now processes data year-by-year:
#   1. obs_fine (training): full 1951–1980 used for every year
#   2. sim_coarse (predict): sliced to one year at a time
#   3. Interpolation + downscaling + saving to .npy happens per year
#   4. Results accumulated in npy_stack
#
# This reduces memory footprint and prevents regridding bottlenecks.

print(f"\n{'='*70}")
print(f"Statistical Downscaling with Year-by-Year Processing")
print(f"{'='*70}")

sd_results = {}

for var in VARIABLES:
    print(f"\n{'='*70}")
    print(f"Variable: {var}")
    print(f"{'='*70}")

    t0_var = time.time()
    
    sd = StatisticalDownscaling(
        variable=var,
        n_iterations=20,
        n_processes=1,  # ← synchronous; regridding & downscaling loop over years
        randomization_seed=42,
    )

    output_path = os.path.join(OUTPUT_DIR, f"{SOURCE_ID}_{MEMBER_ID}_{SCENARIO}_{var}_BCSD_yearloop.nc")
    
    print(f"\nDownscaling {var}...")
    print(f"  obs_fine shape (training, 1951–1980): {obs_raw[var].shape}")
    print(f"  sim_coarse shape (full period):       {bc_result[var].shape}")
    
    # The downscale() method now internally loops over years
    # Regridding and downscaling happen per year, reducing memory load
    sd_out = sd.downscale(
        obs_fine=obs_raw,
        sim_coarse=bc_result[[var]],
        output_path=output_path,
    )

    elapsed_var = time.time() - t0_var
    
    # Verify output file
    if os.path.exists(output_path):
        file_size_mb = os.path.getsize(output_path) / 1e6
        print(f"\n✓ Output file written: {file_size_mb:.1f} MB")
        print(f"✓ Time elapsed: {elapsed_var:.0f}s")
    else:
        print(f"⚠️  Output file NOT found at {output_path}")
    
    sd_results[var] = sd_out

print(f"\n{'='*70}")
print(f"✅ Statistical downscaling COMPLETE")
print(f"{'='*70}")

# Summary: list all output files
print("\nOutput files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".nc") and "BCSD" in f:
        fpath = os.path.join(OUTPUT_DIR, f)
        fsize_mb = os.path.getsize(fpath) / 1e6
        print(f"  {f:<65} {fsize_mb:>8.1f} MB")


Testing with year: 2015
bc_result_test time range: 2015-01-01T12:00:00.000000000 to 2015-12-31T12:00:00.000000000

Downscaling (TEST YEAR 2015): pr
🔧 StatisticalDownscaling initialized for pr
   n_iterations: 20
   n_processes : 32
   randomization_seed: 42
  Starting downscale for pr...
  Input shapes:
    obs_fine: (10958, 890, 619)
    sim_coarse: (365, 34, 36)

🔄 Starting statistical downscaling for pr...
   Converting xarray datasets to iris cubes (in-memory)...
   ⚠  obs_fine has curvilinear (rotated-pole) coords — reprojecting to regular lat/lon grid via scipy griddata...
